# 03 — Annotation Validation

Parse VOC XMLs, validate boxes, verify the split manifest
(seed, counts, country distribution, exclusions). **No silent drops** —
every exclusion is listed with a reason.

In [ ]:
import sys, json
from pathlib import Path
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root / "src"))

In [ ]:
from roadvision.data.annotations import voc_box_to_xyxy, validate_box
print("VOC(1-based incl.) -> torch xyxy example:", voc_box_to_xyxy(10, 20, 50, 60))

In [ ]:
from roadvision.config import load_config
config = load_config(root / "configs" / "experiment.yaml")
raw = root / config["dataset"]["root"]
if raw.exists() and any(raw.iterdir()):
    from roadvision.data.dataset import make_splits
    splits, manifest = make_splits(raw, config.to_dict())
    out = root / "data" / "processed" / "split_manifest.json"
    manifest.save(out)
    print("manifest:", out)
    print(json.dumps(manifest.to_dict(), indent=2)[:4000])
else:
    print("Dataset absent — manifest generation skipped.")

In [ ]:
# Per-class box statistics (needs data)
if raw.exists() and any(raw.iterdir()):
    from collections import Counter
    from roadvision.data.dataset import load_split
    ds, _ = load_split(config["dataset"]["root"], config.to_dict(), "train")
    counter = Counter()
    for i in range(min(len(ds), 500)):   # cap for notebook runtime
        _, target = ds[i]
        for lab in target["labels"].tolist():
            counter[lab] += 1
    inv = {v: k for k, v in ds.class_to_idx.items()}
    for lab, n in sorted(counter.items()):
        print(f"{inv.get(lab, lab)}: {n}")
else:
    print("Dataset absent — class stats skipped.")